# 01: Building the dataset
Goal: one table with one row per country, containing the 2023 HDI class (target), 9 World Bank indicators,
region and the engineered `DominantSector`.

All source files were **downloaded by hand from the official websites** (links below) and saved in `data/raw/`
without any changes. 

In [13]:
import numpy as np
import pandas as pd

RAW = "../data/raw"      # files downloaded from the websites (unchanged)
OUT = "../data"          # final cleaned files

REF_YEAR = 2023          # reference year for HDI and all features
LOOKBACK_YEARS = [2022, 2021]   # if 2023 is missing, use 2022, then 2021 (never a later year)

## 1. UNDP HDI table
- Source: UNDP Human Development Report 2025, *Composite indices complete time series*
  (https://hdr.undp.org/data-center/documentation-and-downloads → "Composite indices time series" CSV)
- Saved as `data/raw/HDR25_composite_indices.csv`
- Reference year: 2023 (column `hdi_2023`). Accessed: 23 Nov 2026

In [14]:
hdr = pd.read_csv(f"{RAW}/HDR25_composite_indices.csv", encoding="latin-1")   # file is not UTF-8
print(hdr.shape)
print(list(hdr.columns[:8]))
print("hdi_2023 present:", "hdi_2023" in hdr.columns)

(206, 1112)
['iso3', 'country', 'hdicode', 'region', 'hdi_rank_2023', 'hdi_1990', 'hdi_1991', 'hdi_1992']
hdi_2023 present: True


In [15]:
hdi = hdr[["iso3", "country", "hdi_2023"]].rename(columns={"hdi_2023": "hdi"})
hdi = hdi[~hdi["iso3"].str.startswith("ZZ")]        # drop regional/group aggregates (codes start with ZZ)
print("rows:", len(hdi))
print("missing HDI:", hdi["hdi"].isna().sum())
print(hdi[hdi["hdi"].isna()])
hdi.head()

rows: 195
missing HDI: 2
    iso3                              country  hdi
90   PRK  Korea (Democratic People's Rep. of)  NaN
115  MCO                               Monaco  NaN


,iso3,country,hdi
0,AFG,Afghanistan,0.496
1,ALB,Albania,0.810
2,DZA,Algeria,0.763
3,AND,Andorra,0.913
4,AGO,Angola,0.616


**Observation:** 195 countries after removing aggregates (iso3 starting with "ZZ").
2 countries have no 2023 HDI (DPR Korea, Monaco); they will be dropped because they have no target label.

## 2. World Bank WDI indicators
- Source: World Bank World Development Indicators, https://data.worldbank.org/indicator/ → search the indicator code
  → **Download → CSV**. Accessed: 25 Nov 2026.
- Each download is a zip with two files we use:
  - `API_<code>_....csv`: one row per country, one column per year (1960 to 2025). Saved as `data/raw/wb_<code>.csv`.
  - `Metadata_Country_....csv`: region of every country (same in every zip). Saved once as `data/raw/wb_country_metadata.csv`.
- The first 4 lines of the data file are a title block, so we read it with `skiprows=4`.
- Year rule: use 2023; if missing, use 2022, then 2021; never a year after 2023. The year actually used is stored.

In [16]:
INDICATORS = {
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",          # GDP growth (annual %)
    "FP.CPI.TOTL.ZG":    "inflation",           # inflation, consumer prices (annual %)
    "SL.UEM.TOTL.ZS":    "unemployment",        # unemployment (% of labour force)
    "SP.DYN.IMRT.IN":    "infant_mortality",    # infant deaths per 1,000 live births
    "EG.ELC.ACCS.ZS":    "electricity_access",  # access to electricity (% of population)
    "IT.NET.USER.ZS":    "internet_users",      # internet users (% of population)
    "NV.AGR.TOTL.ZS":    "agriculture_gdp",     # agriculture value added (% of GDP)
    "NV.IND.TOTL.ZS":    "industry_gdp",        # industry value added (% of GDP)
    "NE.TRD.GNFS.ZS":    "trade_gdp",           # trade (% of GDP)
}

example = pd.read_csv(f"{RAW}/wb_SP.DYN.IMRT.IN.csv", skiprows=4)
example[["Country Name", "Country Code", "2021", "2022", "2023"]].head()

,Country Name,Country Code,2021,2022,2023
0,Aruba,ABW,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,44.321233,44.341994,41.969085
2,Afghanistan,AFG,53.200000,51.600000,49.900000
3,Africa Western and Central,AFW,59.132592,60.096106,57.353682
4,Angola,AGO,34.400000,33.700000,33.000000


In [17]:
wdi = pd.DataFrame()        # value used, one row per country code, one column per indicator
wdi_year = pd.DataFrame()   # year that value came from

for code, name in INDICATORS.items():
    wb = pd.read_csv(f"{RAW}/wb_{code}.csv", skiprows=4).set_index("Country Code")
    value = wb[str(REF_YEAR)].copy()
    year = pd.Series(np.nan, index=wb.index)
    year[value.notna()] = REF_YEAR

    for y in LOOKBACK_YEARS:                         # fill gaps with the most recent earlier year
        use = value.isna() & wb[str(y)].notna()
        value[use] = wb.loc[use, str(y)]
        year[use] = y

    wdi[name] = value
    wdi_year[name] = year
    print(f"{name:20s} countries with a value: {value.notna().sum()}")

wdi.head()

gdp_growth           countries with a value: 258
inflation            countries with a value: 228
unemployment         countries with a value: 235
infant_mortality     countries with a value: 244
electricity_access   countries with a value: 263
internet_users       countries with a value: 217
agriculture_gdp      countries with a value: 245
industry_gdp         countries with a value: 248
trade_gdp            countries with a value: 226


,gdp_growth,inflation,unemployment,infant_mortality,electricity_access,internet_users,agriculture_gdp,industry_gdp,trade_gdp
Country Code,,,,,,,,,
ABW,9.516360,NaN,NaN,NaN,100.000000,NaN,0.039529,10.837942,153.719158
AFE,1.941382,7.399186,7.676548,41.969085,50.667516,27.800000,15.305933,26.467975,56.283721
AFG,2.266944,-4.644709,14.008000,49.900000,85.300000,15.928400,34.743247,13.449823,66.999578
AFW,3.644591,5.221168,3.203355,57.353682,57.064737,38.500000,24.321381,23.244644,NaN
AGO,1.320166,13.644102,14.051000,33.000000,51.100000,37.885899,20.909337,35.701247,53.607013


**Observation:** the counts include World Bank aggregates (World, income groups, regions); these are removed
when we merge with the country list below.

## 3. Country metadata (region)
From `wb_country_metadata.csv`. Aggregates (World, income groups, regional totals) have an empty `Region`, so we keep
only rows where `Region` is filled in.

In [18]:
meta = pd.read_csv(f"{RAW}/wb_country_metadata.csv")
print("all entries:", len(meta))
meta = meta[meta["Region"].notna()]
meta = meta[["Country Code", "TableName", "Region"]].rename(
    columns={"Country Code": "iso3", "TableName": "wb_name", "Region": "region"})
print("real economies:", len(meta))
meta["region"].value_counts()

all entries: 265
real economies: 217


region
Europe & Central Asia         58
Sub-Saharan Africa            48
Latin America & Caribbean     42
East Asia & Pacific           37
Middle East & North Africa    23
South Asia                     6
North America                  3
Name: count, dtype: int64

## 4. Merge HDI + metadata + indicators on ISO3 codes

In [19]:
merged = meta.merge(hdi, on="iso3", how="outer", indicator=True)
only_wb  = merged[merged["_merge"] == "left_only"]
only_hdr = merged[merged["_merge"] == "right_only"]
print("in both:", (merged["_merge"] == "both").sum())
print("HDR countries with no World Bank match:", len(only_hdr), list(only_hdr["country"]))
print("World Bank economies not in HDR:", len(only_wb), list(only_wb["wb_name"]))

data = merged[merged["_merge"] == "both"].drop(columns=["_merge", "wb_name"])
data = data.merge(wdi, left_on="iso3", right_index=True, how="left")

n_before = len(data)
lost = data[data["hdi"].isna()]["country"].tolist()
data = data.dropna(subset=["hdi"]).reset_index(drop=True)
print(f"dropped for missing HDI: {n_before - len(data)} {lost}")
print("final countries:", len(data))

in both: 195
HDR countries with no World Bank match: 0 []
World Bank economies not in HDR: 22 ['Aruba', 'American Samoa', 'Bermuda', 'Channel Islands', 'Curaçao', 'Cayman Islands', 'Faroe Islands', 'Gibraltar', 'Greenland', 'Guam', 'Isle of Man', 'Macao SAR, China', 'St. Martin (French part)', 'Northern Mariana Islands', 'New Caledonia', 'Puerto Rico (US)', 'French Polynesia', 'Sint Maarten (Dutch part)', 'Turks and Caicos Islands', 'British Virgin Islands', 'Virgin Islands (US)', 'Kosovo']
dropped for missing HDI: 2 ['Monaco', "Korea (Democratic People's Rep. of)"]
final countries: 193


**Observation:** all 195 HDR countries matched a World Bank ISO3 code (no manual corrections needed).
22 World Bank economies (territories such as Puerto Rico and Greenland, plus Kosovo and Macao) have no HDI and are
excluded. Monaco and DPR Korea are dropped (no 2023 HDI). Final dataset: 193 countries.

## 5. Target and engineered feature

In [20]:
data["hdi_class"] = np.select([data["hdi"] < 0.550, data["hdi"] < 0.800], [0, 1], default=2)
counts = data["hdi_class"].value_counts().sort_index()
print(pd.DataFrame({"count": counts, "%": (counts / len(data) * 100).round(1)}))

           count     %
hdi_class             
0             26  13.5
1             93  48.2
2             74  38.3


In [21]:
agr, ind = data["agriculture_gdp"], data["industry_gdp"]
serv = 100 - agr - ind
plausible = agr.notna() & ind.notna() & (agr >= 0) & (ind >= 0) & (serv >= 0)

shares = pd.DataFrame({"Agriculture": agr, "Industry": ind, "Services": serv})
data["DominantSector"] = shares.fillna(-1).idxmax(axis=1)   # name of the largest share
data.loc[~plausible, "DominantSector"] = np.nan             # missing / inconsistent -> NaN
data["DominantSector"].value_counts(dropna=False)

DominantSector
Services       176
Industry         8
NaN              7
Agriculture      2
Name: count, dtype: int64

**Observation:** classes are imbalanced: Low 26 (13.5%), Medium/High 93 (48.2%), Very High 74 (38.3%).
DominantSector is almost always "Services" (176 of 186 known values); 8 Industry (mainly oil/mining economies),
2 Agriculture, 7 missing because a sector share is missing. It may be redundant with the numerical shares (as the brief notes).

## 6. Final dataset: counts, missingness, save

In [22]:
NUM = list(INDICATORS.values())
cols = ["iso3", "country", "region", "DominantSector"] + NUM + ["hdi", "hdi_class"]
data = data[cols].sort_values("iso3").reset_index(drop=True)

year_used = wdi_year.loc[data["iso3"]]                   # year of every value, same row order
year_used.index.name = "iso3"
missing_report = pd.DataFrame({
    "% missing": data[NUM].isna().mean().mul(100).round(1),
    "values from 2021-2022": (year_used[NUM] < REF_YEAR).sum(),
})
print("countries:", len(data))
print("missing region:", data["region"].isna().sum(), "| missing DominantSector:", data["DominantSector"].isna().sum())
missing_report

countries: 193
missing region: 0 | missing DominantSector: 7


,% missing,values from 2021-2022
gdp_growth,2.1,1
inflation,7.8,3
unemployment,7.3,3
infant_mortality,1.0,0
electricity_access,0.0,0
internet_users,5.7,3
agriculture_gdp,3.6,3
industry_gdp,3.6,3
trade_gdp,11.9,3


In [23]:
data.to_csv(f"{OUT}/hdi_wdi_2023.csv", index=False)
year_used.to_csv(f"{OUT}/feature_year_used.csv")
print("saved data/hdi_wdi_2023.csv and data/feature_year_used.csv")

saved data/hdi_wdi_2023.csv and data/feature_year_used.csv


In [24]:
print("=== Dataset summary ===")
print(f"HDR countries after removing aggregates: {len(hdi)}")
print(f"Matched with World Bank (ISO3): {(merged['_merge'] == 'both').sum()}, unmatched HDR countries: {len(only_hdr)}")
print(f"Dropped for missing {REF_YEAR} HDI: {lost}")
print(f"Final countries: {len(data)}")
for k, name in enumerate(["Low", "Medium/High", "Very High"]):
    n = (data["hdi_class"] == k).sum()
    print(f"  {name:12s} {n:3d} ({n / len(data):.1%})")
print(f"Missing values per feature: {missing_report['% missing'].min()}% to {missing_report['% missing'].max()}% "
      f"(max: {missing_report['% missing'].idxmax()})")
print(f"Values taken from 2021-2022: {int(missing_report['values from 2021-2022'].sum())} in total")

=== Dataset summary ===
HDR countries after removing aggregates: 195
Matched with World Bank (ISO3): 195, unmatched HDR countries: 0
Dropped for missing 2023 HDI: ['Monaco', "Korea (Democratic People's Rep. of)"]
Final countries: 193
  Low           26 (13.5%)
  Medium/High   93 (48.2%)
  Very High     74 (38.3%)
Missing values per feature: 0.0% to 11.9% (max: trade_gdp)
Values taken from 2021-2022: 19 in total


## Summary
The fixed dataset `data/hdi_wdi_2023.csv` has 193 countries, 9 numerical and 2 categorical predictors, and the
3-class target `hdi_class`. Missing predictor values are kept as NaN on purpose: the imputation methods are compared in
`03a_preprocessing.ipynb`. Sources, codes and rules are documented in `data/data_dictionary.md`.